In [52]:
import pandas as pd
import numpy as np
import glob
import os

PATHS = {
    "AT": {
        "raw":       "../data/AT/raw_data/intraday_epex/trades/",
        "processed": "../data/AT/processed/",
    },
    "DE": {
        "raw":       "../data/DE/raw_data/intraday_epex/trades/",
        "processed": "../data/DE/processed/",
    },
}

for market in PATHS.values():
    os.makedirs(market["processed"], exist_ok=True)

In [ ]:
def parse_filename_meta(filepath: str) -> dict:
    """Extract delivery_start / delivery_end from the CSV filename.

    Filename format: '{delivery_start} {delivery_end}.csv'
    Timestamps use underscores instead of colons, e.g.
      2024-01-01T00_00_00+01_00 2024-01-01T01_00_00+01_00.csv
    """
    name  = os.path.splitext(os.path.basename(filepath))[0]
    parts = name.split(" ")
    if len(parts) == 2:
        return {
            "delivery_start": parts[0].replace("_", ":"),
            "delivery_end":   parts[1].replace("_", ":"),
        }
    return {"delivery_start": None, "delivery_end": None}


def load_raw_trades(raw_path: str, market_label: str) -> pd.DataFrame:

    files = glob.glob(os.path.join(raw_path, "**/*.csv"), recursive=True)

    if not files:
        raise FileNotFoundError(f"No CSV files found under {raw_path}")

    print(f"[{market_label}] Found {len(files)} files — loading...")

    frames = []
    for f in files:
        try:
            chunk = pd.read_csv(f, dtype=str)
            meta  = parse_filename_meta(f)
            chunk["delivery_start"] = meta["delivery_start"]
            chunk["delivery_end"]   = meta["delivery_end"]
            frames.append(chunk)
        except Exception as e:
            print(f"  Skipped {f}: {e}")

    df = pd.concat(frames, ignore_index=True)
    df["market"] = market_label

    print(f"[{market_label}] Raw shape: {df.shape}")
    return df


df_at = load_raw_trades(PATHS["AT"]["raw"], "AT")
df_de = load_raw_trades(PATHS["DE"]["raw"], "DE")

In [ ]:
DOMESTIC_AREAS = {
    "AT": {"10YAT-APG------L"},
    "DE": {
        "10YDE-VE-------2",    # TenneT DE
        "10YDE-RWENET---I",    # Amprion
        "10YDE-EON------1",    # E.ON (now TenneT)
        "10Y1001A1001A63L",    # 50Hertz
    },
}


def clean_trades(df: pd.DataFrame, market_label: str) -> pd.DataFrame:

    df = df.copy()

    # ── 2a. Trade execution timestamp ───────────────────────────────────────
    df["time"] = (
        pd.to_datetime(df["time"], format="ISO8601", utc=True)
          .dt.tz_convert("Europe/Vienna")
    )

    # ── 2b. Delivery window timestamps (from filename) ──────────────────────
    for col in ["delivery_start", "delivery_end"]:
        df[col] = (
            pd.to_datetime(df[col], format="ISO8601", utc=True)
              .dt.tz_convert("Europe/Vienna")
        )

    df["delivery_duration_min"] = (
        (df["delivery_end"] - df["delivery_start"])
        .dt.total_seconds().div(60).astype(int)
    )

    # ── 2c. Numeric columns ─────────────────────────────────────────────────
    df["price"]    = pd.to_numeric(df["price"],    errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

    # ── 2d. Boolean ─────────────────────────────────────────────────────────
    df["self_trade"] = df["self_trade"].str.strip().str.lower() == "true"

    # ── 2e. Drop constant columns ───────────────────────────────────────────
    for col, expected in [("state", "EXECUTED"), ("revision", "1")]:
        unique_vals = df[col].unique()
        if len(unique_vals) == 1 and str(unique_vals[0]) == expected:
            df = df.drop(columns=[col])
            print(f"[{market_label}] Dropped constant column: '{col}' (always '{expected}')")
        else:
            print(f"[{market_label}] Kept '{col}' — has values: {unique_vals}")

    for col in ["currency", "quantity_unit", "exchange"]:
        unique_vals = df[col].unique()
        if len(unique_vals) == 1:
            df = df.drop(columns=[col])
            print(f"[{market_label}] Dropped constant column: '{col}' (always '{unique_vals[0]}')")
        else:
            print(f"[{market_label}] Kept '{col}' — has values: {unique_vals}")

    # ── 2f. Drop unusable rows ──────────────────────────────────────────────
    before = len(df)
    df = df.dropna(subset=["price", "quantity", "time"])
    df = df[df["quantity"] > 0]
    df = df.drop_duplicates(subset=["trade_id"])
    dropped = before - len(df)
    print(f"[{market_label}] Dropped {dropped:,} bad/duplicate rows ({dropped/before:.1%})")

    # ── 2g. Feature engineering ─────────────────────────────────────────────
    df["date"]        = df["time"].dt.date
    df["hour"]        = df["time"].dt.hour
    df["minute"]      = df["time"].dt.minute
    df["day_of_week"] = df["time"].dt.dayofweek
    df["week"]        = df["time"].dt.isocalendar().week.astype(int)
    df["month"]       = df["time"].dt.month
    df["year"]        = df["time"].dt.year
    df["is_weekend"]  = df["day_of_week"] >= 5

    df["cross_border"] = df["buy_delivery_area"] != df["sell_delivery_area"]
    domestic = DOMESTIC_AREAS[market_label]
    df["is_domestic"] = (
        df["buy_delivery_area"].isin(domestic) |
        df["sell_delivery_area"].isin(domestic)
    )
    df["traded_value_eur"] = df["price"] * df["quantity"]

    # Lead time: minutes between trade execution and delivery start
    # Negative values indicate trades made after delivery has begun
    df["lead_time_min"] = (
        (df["delivery_start"] - df["time"])
        .dt.total_seconds().div(60).round().astype(int)
    )

    print(f"[{market_label}] Final shape: {df.shape}\n")
    return df


df_at = clean_trades(df_at, "AT")
df_de = clean_trades(df_de, "DE")

In [ ]:
def sanity_check(df: pd.DataFrame, market_label: str):
    print(f"\n{'='*55}")
    print(f"  SANITY CHECK — {market_label}")
    print(f"{'='*55}")
    print(f"  Rows              : {len(df):,}")
    print(f"  Columns           : {list(df.columns)}")
    print(f"  Date range        : {df['time'].min().date()}  →  {df['time'].max().date()}")
    print(f"  Unique contracts  : {df['contract_id'].nunique():,}")
    print(f"  Price range       : {df['price'].min():.2f}  →  {df['price'].max():.2f} €/MWh")
    print(f"  Negative prices   : {(df['price'] < 0).mean():.2%}")
    print(f"  Cross-border      : {df['cross_border'].mean():.2%}")
    print(f"  Self trades       : {df['self_trade'].mean():.2%}")
    print(f"  Domestic trades   : {df['is_domestic'].mean():.2%}")

    prod = df["delivery_duration_min"].value_counts().sort_index()
    prod_pct = (prod / len(df) * 100).round(1)
    print(f"  Product durations (min → trades):")
    for dur, cnt in prod.items():
        print(f"    {dur:>5} min : {cnt:>10,}  ({prod_pct[dur]:.1f}%)")

    print(f"  Lead time (min)   : "
          f"median {df['lead_time_min'].median():.0f}  "
          f"p5 {df['lead_time_min'].quantile(0.05):.0f}  "
          f"p95 {df['lead_time_min'].quantile(0.95):.0f}")

    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"  Nulls             : none ✓")
    else:
        print(f"  Nulls             :\n{nulls}")
    print()


sanity_check(df_at, "AT")
sanity_check(df_de, "DE")

In [58]:
def save_processed(df: pd.DataFrame, market_label: str):
    out_path = os.path.join(
        PATHS[market_label]["processed"],
        f"intraday_trades_{market_label}.parquet"
    )
    df.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")
    
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"[{market_label}] Saved → {out_path}  ({size_mb:.1f} MB)")


save_processed(df_at, "AT")
save_processed(df_de, "DE")

[AT] Saved → ../data/AT/processed/intraday_trades_AT.parquet  (69.8 MB)
[DE] Saved → ../data/DE/processed/intraday_trades_DE.parquet  (168.5 MB)
